In [55]:
import pandas as pd
import numpy as np

In [56]:
df_summary = pd.read_csv(r"C:\Users\ahmed\Documents\Data Science LUXDEV\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\CSV\CLEAN\forecast_summary.csv")
df_summary.head()

,product_id,product_name,product_category,average_daily_forecast,lead_time_days,safety_stock_days,leadtime_demand,safetystock_units,reorder_point,forecast_14_day_total
0,SKU001,Soda 500ml,Beverages,49.1,5,4,245.5,196.4,441.9,687.4
1,SKU002,Bottled Water 1L,Beverages,50.4,6,4,302.4,201.6,504.0,705.6
2,SKU003,Juice 1L,Beverages,47.4,6,2,284.4,94.8,379.2,663.6
3,SKU004,Energy Drink 250ml,Beverages,36.7,3,4,110.1,146.8,256.9,513.8
4,SKU005,Iced Tea 500ml,Beverages,66.3,6,3,397.8,198.9,596.7,928.2


In [57]:
df_sales = pd.read_csv(r"C:\Users\ahmed\Documents\Data Science LUXDEV\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\CSV\CLEAN\feature_engineered_sales.csv",parse_dates=['date'])
latest_date = df_sales['date'].max()

df_sales = df_sales[df_sales['date']==latest_date]
df_sales =df_sales[['product_id','date','stock_level_end_of_day']]
df_sales = df_sales[['product_id','stock_level_end_of_day']]
df_sales.head()

,product_id,stock_level_end_of_day
730,SKU001,519.2
1461,SKU002,611.6
2192,SKU003,614.0
2923,SKU004,327.7
3654,SKU005,435.7


In [58]:
df = pd.merge(df_summary,df_sales, on= 'product_id', how= 'outer')
df.head()

,product_id,product_name,product_category,average_daily_forecast,lead_time_days,safety_stock_days,leadtime_demand,safetystock_units,reorder_point,forecast_14_day_total,stock_level_end_of_day
0,SKU001,Soda 500ml,Beverages,49.1,5,4,245.5,196.4,441.9,687.4,519.2
1,SKU002,Bottled Water 1L,Beverages,50.4,6,4,302.4,201.6,504.0,705.6,611.6
2,SKU003,Juice 1L,Beverages,47.4,6,2,284.4,94.8,379.2,663.6,614.0
3,SKU004,Energy Drink 250ml,Beverages,36.7,3,4,110.1,146.8,256.9,513.8,327.7
4,SKU005,Iced Tea 500ml,Beverages,66.3,6,3,397.8,198.9,596.7,928.2,435.7


In [59]:
df = df.rename(columns={'stock_level_end_of_day':'current_stock'})
df.head()

,product_id,product_name,product_category,average_daily_forecast,lead_time_days,safety_stock_days,leadtime_demand,safetystock_units,reorder_point,forecast_14_day_total,current_stock
0,SKU001,Soda 500ml,Beverages,49.1,5,4,245.5,196.4,441.9,687.4,519.2
1,SKU002,Bottled Water 1L,Beverages,50.4,6,4,302.4,201.6,504.0,705.6,611.6
2,SKU003,Juice 1L,Beverages,47.4,6,2,284.4,94.8,379.2,663.6,614.0
3,SKU004,Energy Drink 250ml,Beverages,36.7,3,4,110.1,146.8,256.9,513.8,327.7
4,SKU005,Iced Tea 500ml,Beverages,66.3,6,3,397.8,198.9,596.7,928.2,435.7


In [60]:
df['days_of_cover']=round(df['current_stock']/df['average_daily_forecast'],0)
df.head()

,product_id,product_name,product_category,average_daily_forecast,lead_time_days,safety_stock_days,leadtime_demand,safetystock_units,reorder_point,forecast_14_day_total,current_stock,days_of_cover
0,SKU001,Soda 500ml,Beverages,49.1,5,4,245.5,196.4,441.9,687.4,519.2,11.0
1,SKU002,Bottled Water 1L,Beverages,50.4,6,4,302.4,201.6,504.0,705.6,611.6,12.0
2,SKU003,Juice 1L,Beverages,47.4,6,2,284.4,94.8,379.2,663.6,614.0,13.0
3,SKU004,Energy Drink 250ml,Beverages,36.7,3,4,110.1,146.8,256.9,513.8,327.7,9.0
4,SKU005,Iced Tea 500ml,Beverages,66.3,6,3,397.8,198.9,596.7,928.2,435.7,7.0


In [61]:
watchout_buffer  = 0.25

conditions = [
    df['days_of_cover']< df['lead_time_days'],
    df['current_stock']<= df['reorder_point'],
    df['current_stock']<= (df['reorder_point'] * (1+watchout_buffer))
]
labels = ['Critical','reorder_now','watch_out']
df['status']= np.select(conditions,labels,default='OK')

df.head()

,product_id,product_name,product_category,average_daily_forecast,lead_time_days,safety_stock_days,leadtime_demand,safetystock_units,reorder_point,forecast_14_day_total,current_stock,days_of_cover,status
0,SKU001,Soda 500ml,Beverages,49.1,5,4,245.5,196.4,441.9,687.4,519.2,11.0,watch_out
1,SKU002,Bottled Water 1L,Beverages,50.4,6,4,302.4,201.6,504.0,705.6,611.6,12.0,watch_out
2,SKU003,Juice 1L,Beverages,47.4,6,2,284.4,94.8,379.2,663.6,614.0,13.0,OK
3,SKU004,Energy Drink 250ml,Beverages,36.7,3,4,110.1,146.8,256.9,513.8,327.7,9.0,OK
4,SKU005,Iced Tea 500ml,Beverages,66.3,6,3,397.8,198.9,596.7,928.2,435.7,7.0,reorder_now


In [62]:
df['order_upto_level'] =df['leadtime_demand']+df['safety_stock_days']+df['forecast_14_day_total']
df['order_upto_level'] = round(df['order_upto_level'],0)
df.head()

,product_id,product_name,product_category,average_daily_forecast,lead_time_days,safety_stock_days,leadtime_demand,safetystock_units,reorder_point,forecast_14_day_total,current_stock,days_of_cover,status,order_upto_level
0,SKU001,Soda 500ml,Beverages,49.1,5,4,245.5,196.4,441.9,687.4,519.2,11.0,watch_out,937.0
1,SKU002,Bottled Water 1L,Beverages,50.4,6,4,302.4,201.6,504.0,705.6,611.6,12.0,watch_out,1012.0
2,SKU003,Juice 1L,Beverages,47.4,6,2,284.4,94.8,379.2,663.6,614.0,13.0,OK,950.0
3,SKU004,Energy Drink 250ml,Beverages,36.7,3,4,110.1,146.8,256.9,513.8,327.7,9.0,OK,628.0
4,SKU005,Iced Tea 500ml,Beverages,66.3,6,3,397.8,198.9,596.7,928.2,435.7,7.0,reorder_now,1329.0


In [63]:
#difference between where I am currently and where I want to be
df['gap'] = df['order_upto_level'] - df['current_stock']
df.head()

,product_id,product_name,product_category,average_daily_forecast,lead_time_days,safety_stock_days,leadtime_demand,safetystock_units,reorder_point,forecast_14_day_total,current_stock,days_of_cover,status,order_upto_level,gap
0,SKU001,Soda 500ml,Beverages,49.1,5,4,245.5,196.4,441.9,687.4,519.2,11.0,watch_out,937.0,417.8
1,SKU002,Bottled Water 1L,Beverages,50.4,6,4,302.4,201.6,504.0,705.6,611.6,12.0,watch_out,1012.0,400.4
2,SKU003,Juice 1L,Beverages,47.4,6,2,284.4,94.8,379.2,663.6,614.0,13.0,OK,950.0,336.0
3,SKU004,Energy Drink 250ml,Beverages,36.7,3,4,110.1,146.8,256.9,513.8,327.7,9.0,OK,628.0,300.3
4,SKU005,Iced Tea 500ml,Beverages,66.3,6,3,397.8,198.9,596.7,928.2,435.7,7.0,reorder_now,1329.0,893.3


In [64]:
#Now column for suggested order quantity(we only want to reorder a calculated quantity with immediate effect SKU'S whose status are 'reorder now'  or 'critical')
#we are using ceil instead of round , because we want to reorder to the nearest whole number ,
#and again if the gap is 893.4 for instance, we cant reorder 893, it would then be less quantity so we ceil it up to 894.

df['suggested_order_quantity'] = np.where(df['status'].isin(['reorder_now','critical']),np.ceil(df['gap']),0).astype(int)
df

,product_id,product_name,product_category,average_daily_forecast,lead_time_days,safety_stock_days,leadtime_demand,safetystock_units,reorder_point,forecast_14_day_total,current_stock,days_of_cover,status,order_upto_level,gap,suggested_order_quantity
0,SKU001,Soda 500ml,Beverages,49.1,5,4,245.5,196.4,441.9,687.4,519.2,11.0,watch_out,937.0,417.8,0
1,SKU002,Bottled Water 1L,Beverages,50.4,6,4,302.4,201.6,504.0,705.6,611.6,12.0,watch_out,1012.0,400.4,0
2,SKU003,Juice 1L,Beverages,47.4,6,2,284.4,94.8,379.2,663.6,614.0,13.0,OK,950.0,336.0,0
3,SKU004,Energy Drink 250ml,Beverages,36.7,3,4,110.1,146.8,256.9,513.8,327.7,9.0,OK,628.0,300.3,0
4,SKU005,Iced Tea 500ml,Beverages,66.3,6,3,397.8,198.9,596.7,928.2,435.7,7.0,reorder_now,1329.0,893.3,894
5,SKU006,Potato Crisps,Snacks,37.3,9,2,335.7,74.6,410.3,522.2,69.3,2.0,Critical,860.0,790.7,0
6,SKU007,Chocolate Bar,Snacks,53.4,3,4,160.2,213.6,373.8,747.6,425.7,8.0,watch_out,912.0,486.3,0
7,SKU008,Biscuits Pack,Snacks,63.7,5,4,318.5,254.8,573.3,891.8,595.4,9.0,watch_out,1214.0,618.6,0
8,SKU009,Peanuts 200g,Snacks,74.3,4,3,297.2,222.9,520.1,1040.2,303.3,4.0,reorder_now,1340.0,1036.7,1037
9,SKU010,Popcorn Bag,Snacks,37.4,7,4,261.8,149.6,411.4,523.6,257.5,7.0,reorder_now,789.0,531.5,532


In [65]:
df.shape

(25, 16)

In [66]:
df.to_csv(r"C:\Users\ahmed\Documents\Data Science LUXDEV\PPORTFOLIO PROJECTS\INVENTORY FORECASTING\CSV\CLEAN\reorders.csv",index=False)